<a href="https://colab.research.google.com/github/emilymhudson/mime-forensic-log-parser/blob/main/Log_Parser_Dev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
%%writefile advanced_parser.py
import re
import json
import email
import hashlib
from email import policy
from bs4 import BeautifulSoup
from urllib.parse import urlparse

class MIMETelemetryEngine:
    """
    Advanced deterministic MIME parsing engine designed for multi-modal feature extraction.
    Decomposes RFC 5322 formatted emails to isolate unstructured text, network topology,
    and binary artifacts for downstream ingestion into deep learning fusion architectures.
    """

    def __init__(self, raw_eml_string):
        self.msg = email.message_from_string(raw_eml_string, policy=policy.default)
        self.telemetry = {
            "network_topology": {},
            "linguistic_features": {},
            "binary_artifacts": [],
            "extracted_routing_nodes": [],
            "obfuscation_flags": []
        }

    def _extract_authentication_topology(self):
        """Maps authentication headers and extracts IP routing nodes."""
        auth_headers = ["received-spf", "authentication-results"]
        for header, value in self.msg.items():
            if header.lower() in auth_headers:
                spf = re.search(r'spf=(\w+)', value, re.IGNORECASE)
                dkim = re.search(r'dkim=(\w+)', value, re.IGNORECASE)
                dmarc = re.search(r'dmarc=(\w+)', value, re.IGNORECASE)

                self.telemetry["network_topology"]["SPF"] = spf.group(1) if spf else "none"
                self.telemetry["network_topology"]["DKIM"] = dkim.group(1) if dkim else "none"
                self.telemetry["network_topology"]["DMARC"] = dmarc.group(1) if dmarc else "none"

            # Extract IPv4 nodes for graph analytics
            if header.lower() == "received":
                ips = re.findall(r'\b(?:[0-9]{1,3}\.){3}[0-9]{1,3}\b', value)
                self.telemetry["extracted_routing_nodes"].extend(ips)

        self.telemetry["extracted_routing_nodes"] = list(set(self.telemetry["extracted_routing_nodes"]))

    def _detect_linguistic_obfuscation(self, raw_text):
        """
        Heuristic detection of adversarial text manipulation designed to bypass SEGs.
        Flags zero-width spaces, homoglyphs, and excessive HTML rendering anomalies.
        """
        zero_width_chars = r'[\u200B-\u200D\uFEFF]'
        if re.search(zero_width_chars, raw_text):
            self.telemetry["obfuscation_flags"].append("ZERO_WIDTH_CHARACTER_INJECTION")

        if len(re.findall(r'https?://', raw_text)) > 5:
            self.telemetry["obfuscation_flags"].append("HIGH_DENSITY_URL_EMBEDDING")

    def _isolate_binary_artifacts(self, part):
        """Hashes and isolates binary payloads (images/audio) for convolutional/transformer analysis."""
        payload = part.get_payload(decode=True)
        if payload:
            artifact_hash = hashlib.sha256(payload).hexdigest()
            self.telemetry["binary_artifacts"].append({
                "filename": part.get_filename() or "unknown_artifact",
                "content_type": part.get_content_type(),
                "size_bytes": len(payload),
                "sha256": artifact_hash
            })

    def decompose_payload(self):
        """Execution method: Walks the MIME tree and structures the data tensor mapping."""
        self._extract_authentication_topology()

        body_text = ""
        for part in self.msg.walk():
            content_type = part.get_content_type()
            content_disposition = str(part.get("Content-Disposition"))

            # Isolate Textual Modalities (Strip HTML for DistilBERT Tokenization prep)
            if content_type == "text/plain" and "attachment" not in content_disposition:
                body_text += part.get_payload(decode=True).decode(errors='ignore')
            elif content_type == "text/html" and "attachment" not in content_disposition:
                html_payload = part.get_payload(decode=True).decode(errors='ignore')
                soup = BeautifulSoup(html_payload, "html.parser")
                body_text += soup.get_text(separator=" ")

            # Isolate Visual/Acoustic Modalities
            elif "attachment" in content_disposition or part.get_filename():
                self._isolate_binary_artifacts(part)

        # Normalize text and run heuristic obfuscation checks
        normalized_text = re.sub(r'\s+', ' ', body_text).strip()
        self._detect_linguistic_obfuscation(normalized_text)

        # Extract and sanitize URLs
        urls = re.findall(r'https?://[^\s<>"]+|www\.[^\s<>"]+', normalized_text)
        self.telemetry["linguistic_features"] = {
            "tokenization_length": len(normalized_text.split()),
            "embedded_domains": list(set([urlparse(url).netloc for url in urls])),
            "raw_normalized_text": normalized_text
        }

        return json.dumps(self.telemetry, indent=4)


# MOCK EXECUTION FOR ARCHITECTURE VALIDATION

mock_adversarial_eml = """Received: from static.attacker.net (192.168.1.105)
Received-SPF: fail (domain of alert@aws-verification.com does not designate 192.168.1.105 as permitted sender)
Subject: Urgent: Verify AWS Access
From: AWS Corporate Cloud <alert@aws-verification.com>
Content-Type: multipart/mixed; boundary="boundary_12345"

--boundary_12345
Content-Type: text/html; charset="UTF-8"

<html><body>
Urgent: Your AWS root keys expire in 24 hours.&#8203;
Click <a href="http://cloned-aws-signin-portal.net/login/auth.php">here</a> to verify.
</body></html>

--boundary_12345
Content-Type: image/jpeg; name="tracking_pixel.jpg"
Content-Disposition: attachment; filename="tracking_pixel.jpg"

[FAKE_BINARY_DATA_FOR_SIMULATION]
--boundary_12345--
"""

if __name__ == "__main__":
    print("Initializing MIMETelemetryEngine...")
    engine = MIMETelemetryEngine(mock_adversarial_eml)
    fusion_tensor_data = engine.decompose_payload()
    print("Feature Extraction Complete. Late Fusion JSON Schema Generated:\n")
    print(fusion_tensor_data)

    with open('fusion_telemetry_schema.json', 'w') as f:
        f.write(fusion_tensor_data)

Overwriting advanced_parser.py


In [22]:
!python advanced_parser.py

Initializing MIMETelemetryEngine...
Feature Extraction Complete. Late Fusion JSON Schema Generated:

{
    "network_topology": {
        "SPF": "none",
        "DKIM": "none",
        "DMARC": "none"
    },
    "linguistic_features": {
        "tokenization_length": 13,
        "embedded_domains": [],
        "raw_normalized_text": "Urgent: Your AWS root keys expire in 24 hours.\u200b Click here to verify."
    },
    "binary_artifacts": [
        {
            "filename": "tracking_pixel.jpg",
            "content_type": "image/jpeg",
            "size_bytes": 33,
            "sha256": "ac7c23c473f2a854a1824463cfd946b9333e15c4fcee77ae7c06e96dd80850e5"
        }
    ],
    "extracted_routing_nodes": [
        "192.168.1.105"
    ],
    "obfuscation_flags": [
        "ZERO_WIDTH_CHARACTER_INJECTION"
    ]
}
